# Controle de custos de obra com SQL: o orçamento acertou?

10 perguntas de diretor respondidas em SQL, com dados reais de 8 obras de uma construtora australiana (2011–2015, valores em A\$). Contexto, conclusão e próximos passos estão no [README](README.md).

**Saldo = orçado − real**, como extrato de banco: positivo = sobrou, negativo = estourou.

*Ferramentas: Python, SQL e IA como copiloto (Claude).*

In [1]:
import sqlite3
import pandas as pd

PASTA = 'data/'   # no Colab: suba a pasta data/ ou ajuste o caminho aqui

conn = sqlite3.connect(':memory:')
for tabela in ['obras', 'itens', 'evolucao_mensal']:
    pd.read_csv(PASTA + tabela + '.csv').to_sql(tabela, conn, index=False)

def sql(consulta):
    return pd.read_sql(consulta, conn)

## P1. Somando as 8 obras, o gasto bateu com o orçado?

In [2]:
sql("""
SELECT COUNT(*)                                   AS obras,
       ROUND(SUM(orcado))                         AS orcado,
       ROUND(SUM(custo_real))                     AS custo_real,
       ROUND(SUM(orcado) - SUM(custo_real))       AS saldo,
       ROUND(100.0 * (SUM(orcado) - SUM(custo_real)) / SUM(orcado), 1) AS saldo_pct
FROM obras
""")

,obras,orcado,custo_real,saldo,saldo_pct
0,8,12646423.0,11593773.0,1052650.0,8.3


*Orçaram A\$ 12,6 mi e gastaram A\$ 11,6 mi: saldo de +A\$ 1,05 mi (+8,3%). No total, parece sucesso.*

## P2. Quais obras estouraram e quais sobraram?

In [3]:
sql("""
SELECT obra,
       nome,
       ROUND(orcado)                 AS orcado,
       ROUND(custo_real)             AS custo_real,
       ROUND(saldo)                  AS saldo,
       ROUND(100.0 * saldo_pct, 1)   AS saldo_pct
FROM obras
ORDER BY saldo_pct
""")

,obra,nome,orcado,custo_real,saldo,saldo_pct
0,P06,Reconstrução de via urbana,2821503.0,2845414.0,-23911.0,-0.8
1,P17,Loteamento de marina,1780592.0,1773336.0,7256.0,0.4
2,P04,Recuperação de pavimento arterial,675766.0,646661.0,29105.0,4.3
3,P01,Estacionamento de aeroporto,1635227.0,1546739.0,88488.0,5.4
4,P02,Célula de aterro sanitário,926099.0,857916.0,68182.0,7.4
5,P05,Reconstrução de via semiurbana,1019392.0,922562.0,96831.0,9.5
6,P03,Rodovia de contorno regional,1456829.0,1295759.0,161070.0,11.1
7,P18,Recuperação de estrada rural,2331015.0,1705386.0,625629.0,26.8


*Só a via urbana estourou (−0,8%). A estrada rural sobrou 26,8% (A\$ 626 mil), mais da metade do saldo total.*

## P3. Quantos itens passaram do orçado?

Tolerância zero: qualquer real acima do orçado conta como estouro.

In [4]:
sql("""
SELECT situacao,
       COUNT(*)                                                   AS itens,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM itens), 1)  AS pct_itens,
       ROUND(SUM(saldo))                                          AS saldo
FROM itens
GROUP BY situacao
ORDER BY itens DESC
""")

,situacao,itens,pct_itens,saldo
0,Abaixo do orçado,130,49.1,3432708.0
1,Estourou,92,34.7,-2322700.0
2,Não orçado,30,11.3,-175025.0
3,Não executado,13,4.9,117666.0


*122 de 265 itens (46%) custaram mais que o previsto: 92 estouraram e 30 nem estavam no orçamento. Os estouros (−A\$ 2,3 mi) foram compensados pelas sobras (+A\$ 3,4 mi).*

## P4. Em que tipo de serviço está o erro?

In [5]:
sql("""
SELECT grupo,
       ROUND(SUM(orcado))                             AS orcado,
       ROUND(SUM(custo_real))                         AS custo_real,
       ROUND(SUM(saldo))                              AS saldo,
       ROUND(100.0 * SUM(saldo) / SUM(orcado), 1)     AS saldo_pct
FROM itens
GROUP BY grupo
ORDER BY saldo
""")

,grupo,orcado,custo_real,saldo,saldo_pct
0,Terraplenagem,1406668.0,1738270.0,-331602.0,-23.6
1,Custos indiretos,1485145.0,1584358.0,-99214.0,-6.7
2,Serviços extras e retrabalho,82684.0,148461.0,-65777.0,-79.6
3,"Concreto, meio-fio e calçadas",324807.0,377858.0,-53051.0,-16.3
4,Limpeza e solo vegetal,121924.0,168247.0,-46323.0,-38.0
5,Topografia e qualidade,401686.0,378333.0,23353.0,5.8
6,Impermeabilização (aterro),406239.0,368494.0,37745.0,9.3
7,"Redes (água, esgoto, elétrica)",956327.0,910858.0,45468.0,4.8
8,"Paisagismo, cercas e muros",1080029.0,1005015.0,75014.0,6.9
9,Demolição,131431.0,33260.0,98171.0,74.7


*Terraplenagem estourou A\$ 332 mil (−24%). Pavimentação sobrou A\$ 735 mil (+15%) e tráfego gastou só um terço do previsto (+67%).*

## P5. Quais itens mais estouraram?

In [6]:
sql("""
SELECT i.obra,
       o.nome,
       i.descricao_pt          AS item,
       ROUND(i.orcado)         AS orcado,
       ROUND(i.custo_real)     AS custo_real,
       ROUND(i.saldo)          AS saldo
FROM itens AS i
JOIN obras AS o ON o.obra = i.obra
ORDER BY i.saldo
LIMIT 10
""")

,obra,nome,item,orcado,custo_real,saldo
0,P06,Reconstrução de via urbana,Corte com bota-fora,129418.0,308768.0,-179350.0
1,P03,Rodovia de contorno regional,Preparo do subleito,37229.0,213778.0,-176548.0
2,P18,Recuperação de estrada rural,Corte com bota-fora,236016.0,359268.0,-123252.0
3,P06,Reconstrução de via urbana,Imprimação / tratamento superficial,152157.0,274290.0,-122133.0
4,P06,Reconstrução de via urbana,Base de brita compactada,282664.0,382853.0,-100188.0
5,P06,Reconstrução de via urbana,Troca de solo inadequado,138017.0,215667.0,-77651.0
6,P18,Recuperação de estrada rural,Gerenciamento do projeto,48000.0,123128.0,-75128.0
7,P01,Estacionamento de aeroporto,Aterro com material estocado,94981.0,164871.0,-69890.0
8,P05,Reconstrução de via semiurbana,Raspagem de solo vegetal,7980.0,72746.0,-64767.0
9,P03,Rodovia de contorno regional,Imprimação / tratamento superficial,217171.0,277418.0,-60246.0


*4 dos 10 maiores estouros são da via urbana, quase todos de solo e base. O 1º é o corte com bota-fora (−A\$ 179 mil).*

## P6. O que apareceu na obra sem estar no orçamento?

In [7]:
sql("""
SELECT obra,
       COUNT(*)                          AS itens_nao_orcados,
       ROUND(SUM(custo_real))            AS custo_nao_orcado,
       GROUP_CONCAT(descricao_pt, ', ')  AS itens
FROM itens
WHERE situacao = 'Não orçado'
GROUP BY obra
ORDER BY custo_nao_orcado DESC
""")

,obra,itens_nao_orcados,custo_nao_orcado,itens
0,P06,12,51900.0,"Localização de interferências, Aterro com mate..."
1,P04,4,50267.0,"Ensaios de qualidade, Base de brita compactada..."
2,P17,8,44220.0,"Demolições, Troca de solo inadequado, Barreira..."
3,P02,2,9348.0,"Serviços por administração, Desmobilização e l..."
4,P01,2,8905.0,"Localização de interferências, Paisagismo (sub..."
5,P05,1,5849.0,Ferramentas pequenas
6,P18,1,4536.0,Ensaios de qualidade


*30 itens, A\$ 175 mil. A via urbana teve 12, entre eles retrabalho, paralisação por chuva e alojamento.*

## P7. Onde a terraplenagem estourou?

`HAVING` tira as obras que não orçaram terraplenagem (orçado zero).

In [8]:
sql("""
SELECT o.nome,
       ROUND(SUM(i.orcado))                              AS orcado_terraplenagem,
       ROUND(SUM(i.custo_real))                          AS real_terraplenagem,
       ROUND(100.0 * SUM(i.saldo) / SUM(i.orcado), 1)    AS saldo_pct
FROM itens AS i
JOIN obras AS o ON o.obra = i.obra
WHERE i.grupo = 'Terraplenagem'
GROUP BY o.nome
HAVING SUM(i.orcado) > 0
ORDER BY saldo_pct
""")

,nome,orcado_terraplenagem,real_terraplenagem,saldo_pct
0,Loteamento de marina,31995.0,73255.0,-129.0
1,Reconstrução de via urbana,267435.0,524629.0,-96.2
2,Estacionamento de aeroporto,176220.0,255009.0,-44.7
3,Célula de aterro sanitário,411809.0,424025.0,-3.0
4,Recuperação de estrada rural,399009.0,395756.0,0.8
5,Reconstrução de via semiurbana,68993.0,43727.0,36.6
6,Rodovia de contorno regional,51207.0,21869.0,57.3


*Na marina custou mais que o dobro (−129%) e na via urbana quase o dobro (−96%). Onde a terraplenagem pesava mais no orçamento (aterro e estrada rural), bateu com o previsto.*

## P8. Quais itens mais sobraram?

In [9]:
sql("""
SELECT i.obra,
       o.nome,
       i.descricao_pt          AS item,
       ROUND(i.orcado)         AS orcado,
       ROUND(i.custo_real)     AS custo_real,
       ROUND(i.saldo)          AS saldo
FROM itens AS i
JOIN obras AS o ON o.obra = i.obra
ORDER BY i.saldo DESC
LIMIT 10
""")

,obra,nome,item,orcado,custo_real,saldo
0,P18,Recuperação de estrada rural,Base de brita compactada,408945.0,112261.0,296684.0
1,P04,Recuperação de pavimento arterial,Estabilização de solo in situ,438809.0,223782.0,215027.0
2,P06,Reconstrução de via urbana,Controladores de tráfego,195103.0,38276.0,156827.0
3,P18,Recuperação de estrada rural,Preparo do subleito,131923.0,3484.0,128439.0
4,P18,Recuperação de estrada rural,Troca de solo inadequado,162993.0,36488.0,126505.0
5,P18,Recuperação de estrada rural,Entrega de brita para base,307979.0,187161.0,120818.0
6,P03,Rodovia de contorno regional,Controladores de tráfego,115917.0,2466.0,113450.0
7,P06,Reconstrução de via urbana,Asfalto (CBUQ),374979.0,261664.0,113315.0
8,P03,Rodovia de contorno regional,Asfalto (CBUQ),296793.0,202459.0,94334.0
9,P01,Estacionamento de aeroporto,Asfalto (CBUQ),392817.0,307649.0,85168.0


*4 das 10 maiores sobras são da estrada rural, todas em base e solo (cerca de A\$ 670 mil). Controladores de tráfego aparecem 2 vezes.*

## P9. O erro veio do preço ou da quantidade?

Cada saldo é separado em quantidade (fez mais ou menos que o previsto) e preço (pagou mais ou menos por unidade). Somo o tamanho de cada parte, sem sinal. Ficam de fora 46 itens sem quantidade para comparar.

In [10]:
sql("""
SELECT COUNT(*)                                                     AS itens,
       ROUND(SUM(ABS(efeito_preco)))                                AS tamanho_preco,
       ROUND(SUM(ABS(efeito_quantidade)))                           AS tamanho_quantidade,
       ROUND(100.0 * SUM(ABS(efeito_preco))
             / (SUM(ABS(efeito_preco)) + SUM(ABS(efeito_quantidade)))) AS pct_preco
FROM itens
WHERE efeito_quantidade IS NOT NULL
""")

,itens,tamanho_preco,tamanho_quantidade,pct_preco
0,219,5046904.0,1804505.0,74.0


*Cerca de 3/4 do erro (em dinheiro) veio do preço unitário e 1/4 da quantidade. As quantidades estavam certas; o que errou foi o preço.*

## P10. Onde está o peso de cada obra?

`SUM(...) OVER (PARTITION BY obra)` calcula o total de cada obra sem esmagar as linhas, e `RANK` mostra os 3 grupos que mais pesam.

In [11]:
sql("""
SELECT *
FROM (
    SELECT obra,
           grupo,
           ROUND(SUM(orcado)) AS orcado,
           ROUND(100.0 * SUM(orcado) / SUM(SUM(orcado)) OVER (PARTITION BY obra), 1) AS pct_da_obra,
           RANK() OVER (PARTITION BY obra ORDER BY SUM(orcado) DESC)                 AS posicao
    FROM itens
    GROUP BY obra, grupo
)
WHERE posicao <= 3
ORDER BY obra, posicao
""")

,obra,grupo,orcado,pct_da_obra,posicao
0,P01,Pavimentação,665484.0,40.7,1
1,P01,"Redes (água, esgoto, elétrica)",334030.0,20.4,2
2,P01,Terraplenagem,176220.0,10.8,3
3,P02,Terraplenagem,411809.0,44.5,1
4,P02,Impermeabilização (aterro),227005.0,24.5,2
5,P02,Custos indiretos,138113.0,14.9,3
6,P03,Pavimentação,840632.0,57.7,1
7,P03,Custos indiretos,197699.0,13.6,2
8,P03,"Tráfego, segurança e meio ambiente",120459.0,8.3,3
9,P04,Pavimentação,528842.0,78.3,1


*Pavimentação é o maior grupo em 6 das 8 obras (até 78% do orçamento). É a Curva ABC da obra: o controle começa pelos grupos que concentram o dinheiro.*

## Conclusão
- **No total, sucesso; item a item, não:** saldo de +8,3%, mas 46% dos itens estouraram.
- **O erro está no preço:** cerca de 3/4 veio do custo unitário. O orçamento não está aprendendo com as compras.
- **Erro repetido × imprevisível:** pavimentação e tráfego sobram quase sempre (corrigir a composição); terraplenagem estoura em umas obras e sobra em outras (investigar o solo).